# Contribution 3: Đánh Giá Độ Bền Nhiễu NISQ (NSL-KDD)

**Pipeline:** `Cleaned.csv` → `selector.joblib` → `pca.joblib` → `scaler.joblib` → QSVM

**Mô hình nhiễu:** Depolarizing · Amplitude Damping · Readout Error

**Phân tích:** Graceful degradation · So sánh với SVM cổ điển · Zero-Noise Extrapolation (ZNE)

**Thứ tự thực thi:**
1. Config & Imports
2. Load Transformers & Pipeline
3. Load Datasets
4. Build Noise Models & Kernel
5. Classical SVM Baseline
6. QSVM Noiseless Baseline
7. Depolarizing Noise Scan
8. Amplitude Damping & Readout Error
9. Zero-Noise Extrapolation (ZNE)
10. Per-Category Analysis
11. Visualization
12. Lưu Kết Quả
13. Summary

## 1. Config & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import os
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC
from sklearn.metrics import (f1_score, accuracy_score, classification_report,
                              confusion_matrix)
from sklearn.preprocessing import LabelEncoder

from qiskit.circuit.library import zz_feature_map
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (NoiseModel, depolarizing_error,
                               amplitude_damping_error, ReadoutError)
from qiskit_aer.primitives import SamplerV2 as AerSamplerV2
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.algorithms import QSVC

import qiskit, qiskit_aer, qiskit_machine_learning
print(f'Qiskit              : {qiskit.__version__}')
print(f'Qiskit Aer          : {qiskit_aer.__version__}')
print(f'Qiskit ML           : {qiskit_machine_learning.__version__}')

# ── Cấu hình chính ──
RANDOM_STATE  = 42
N_QUBITS      = 4        # khớp với pca_4components.joblib
REPS          = 2        # ZZFeatureMap reps
ANGLE_MAX     = np.pi    # MinMaxScaler range [0, pi]

LABEL_COLS    = ['label', 'label_binary', 'label_multiclass', 'attack_category']

# Đường dẫn
DATA_DIR      = '../data/processed_data'
MODELS_DIR    = '../models'
OUTPUT_DIR    = '../data/processed_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Train/test sizes
TRAIN_SIZES      = [100, 200, 500, 1000]
TEST_SIZES       = [100]          # test set cho QSVM comparison
TEST_SIZES_SVM   = [100, 200, 300]  # test sets cho SVM baseline (đánh giá độ tin cậy)

# QSVM train sizes — các size có sẵn model sẽ được load, chưa có sẽ train & lưu
QSVM_TRAIN_SIZES = [100]   # ← chỉnh tại đây

QSVM_MODEL_DIR   = f'{MODELS_DIR}/qsvm_cache'
os.makedirs(QSVM_MODEL_DIR, exist_ok=True)

# Mức nhiễu depolarizing để quét
NOISE_LEVELS  = [0.0, 0.005, 0.01, 0.02, 0.05, 0.08, 0.10]

# Mức nhiễu đại diện cho per-category analysis (phải là subset của NOISE_LEVELS)
NOISE_REPR    = [0.01, 0.05, 0.10]

# ZNE scale factors
ZNE_SCALES    = [1, 2, 3]
ZNE_P1Q       = 0.02

np.random.seed(RANDOM_STATE)
print('\nConfig OK')
print(f'  N_QUBITS         : {N_QUBITS}')
print(f'  REPS             : {REPS}')
print(f'  Train sizes      : {TRAIN_SIZES}')
print(f'  Test sizes QSVM  : {TEST_SIZES}')
print(f'  Test sizes SVM   : {TEST_SIZES_SVM}')
print(f'  Noise levels     : {NOISE_LEVELS}')
print(f'  Noise repr       : {NOISE_REPR}')


## 2. Load Transformers & Transform Pipeline

In [ ]:
# Load 3 transformer đã fit trên full train
selector = joblib.load(f'{MODELS_DIR}/feature_selector_k20.joblib')
pca      = joblib.load(f'{MODELS_DIR}/pca_4components.joblib')
scaler   = joblib.load(f'{MODELS_DIR}/scaler_minmax_pi.joblib')

print(f'SelectKBest : k={selector.k}, score_func={selector.score_func.__name__}')
print(f'PCA         : n_components={pca.n_components_}, '
      f'variance={pca.explained_variance_ratio_.sum()*100:.2f}%')
print(f'Scaler      : range=[{scaler.feature_range[0]:.4f}, {scaler.feature_range[1]:.4f}]')


def transform_pipeline(df, feature_cols):
    """Cleaned CSV -> angle-encoded numpy array. KHÔNG fit lại."""
    X_raw = df[feature_cols].to_numpy(dtype=np.float32)
    X_sel = selector.transform(X_raw)
    X_pca = pca.transform(X_sel)
    X_ang = np.clip(scaler.transform(X_pca), 0, ANGLE_MAX)
    return X_ang


## 3. Load Datasets

In [ ]:
# ── Load sample subsets ──
datasets     = {}
feature_cols = None

for n in TRAIN_SIZES:
    path = f'{DATA_DIR}/NSL_KDD_Train_Sample{n}.csv'
    if os.path.exists(path):
        df = pd.read_csv(path)
        if feature_cols is None:
            feature_cols = [c for c in df.columns if c not in LABEL_COLS]
        datasets[f'train_{n}'] = {
            'X': transform_pipeline(df, feature_cols),
            'y': df['label_binary'].to_numpy(dtype=np.int64),
            'meta': df[LABEL_COLS].copy()
        }
        print(f'  Train {n:>4}: {datasets[f"train_{n}"]["X"].shape}')

# Load tất cả test sizes cần thiết (QSVM + SVM)
all_test_sizes = sorted(set(TEST_SIZES + TEST_SIZES_SVM))
for n in all_test_sizes:
    path = f'{DATA_DIR}/NSL_KDD_Test_Sample{n}.csv'
    if os.path.exists(path):
        df = pd.read_csv(path)
        datasets[f'test_{n}'] = {
            'X': transform_pipeline(df, feature_cols),
            'y': df['label_binary'].to_numpy(dtype=np.int64),
            'meta': df[LABEL_COLS].copy()
        }
        print(f'  Test  {n:>4}: {datasets[f"test_{n}"]["X"].shape}')
    else:
        print(f'  [MISSING] NSL_KDD_Test_Sample{n}.csv')

# Test set cố định cho QSVM comparison
TEST_KEY_FIXED = f'test_{TEST_SIZES[0]}'
X_test_fixed   = datasets[TEST_KEY_FIXED]['X']
y_test_fixed   = datasets[TEST_KEY_FIXED]['y']

print(f'\nfeature_cols     : {len(feature_cols)} cột')
print(f'Test cố định QSVM: {TEST_KEY_FIXED} — {X_test_fixed.shape}')
print('\nHead X_train_100 (5 dòng đầu):')
print(datasets['train_100']['X'][:5])


## 4. Build Noise Models & Kernel

In [ ]:
def build_noise_model(noise_type='depolarizing', p1q=0.01, p2q=None,
                      gamma_ad=0.01, readout_err=0.02):
    """
    Tạo NoiseModel theo loại và mức nhiễu.
    Parameters
    ----------
    noise_type  : 'depolarizing' | 'amplitude_damping' | 'readout' | 'combined'
    p1q         : xác suất lỗi 1-qubit gate (depolarizing)
    p2q         : xác suất lỗi 2-qubit gate; nếu None thì = 5*p1q (phản ánh NISQ reality)
    gamma_ad    : amplitude damping parameter
    readout_err : xác suất đọc sai
    """
    if p2q is None:
        p2q = min(5 * p1q, 1.0)

    nm = NoiseModel()

    if noise_type in ('depolarizing', 'combined'):
        if p1q > 0:
            nm.add_all_qubit_quantum_error(depolarizing_error(p1q, 1),
                                           ['u', 'rx', 'ry', 'rz', 'h', 'p'])
        if p2q > 0:
            nm.add_all_qubit_quantum_error(depolarizing_error(p2q, 2),
                                           ['cx', 'cz', 'ecr'])

    if noise_type in ('amplitude_damping', 'combined'):
        if gamma_ad > 0:
            nm.add_all_qubit_quantum_error(amplitude_damping_error(gamma_ad),
                                           ['u', 'rx', 'ry', 'rz'])

    if noise_type in ('readout', 'combined'):
        if readout_err > 0:
            nm.add_all_qubit_readout_error(
                ReadoutError([[1 - readout_err, readout_err],
                              [readout_err, 1 - readout_err]])
            )

    return nm


def build_qsvm_kernel(noise_model=None, shots=1024):
    """Tạo FidelityQuantumKernel với ZZFeatureMap 4-qubit."""
    fm = zz_feature_map(feature_dimension=N_QUBITS, reps=REPS)

    backend_options = {'noise_model': noise_model} if noise_model is not None else None
    sampler  = AerSamplerV2(default_shots=shots, options=backend_options)
    fidelity = ComputeUncompute(sampler=sampler)
    return FidelityQuantumKernel(feature_map=fm, fidelity=fidelity)


# Kiểm tra nhanh
print('=== Test noise models ===')
for nt in ['depolarizing', 'amplitude_damping', 'readout', 'combined']:
    nm = build_noise_model(noise_type=nt, p1q=0.01, gamma_ad=0.01, readout_err=0.02)
    print(f'  [{nt:<18}] basis_gates={nm.basis_gates[:4]}...')

print('\n=== Test kernel build (noiseless) ===')
kernel_test = build_qsvm_kernel(noise_model=None)
X_tiny = datasets['train_100']['X'][:4]
K_tiny = kernel_test.evaluate(X_tiny, X_tiny)
print(f'  Kernel matrix 4x4: diag={np.diag(K_tiny).round(3)}')
print('  Build OK')


## 5. Classical SVM Baseline (RBF kernel)

In [ ]:
# Classical SVM RBF — test trên nhiều test sizes để đánh giá độ tin cậy
# QSVM comparison vẫn dùng test_100 cố định

svm_baselines = {}  # lưu kết quả test_100 để so sánh với QSVM

for test_key in [f'test_{n}' for n in TEST_SIZES_SVM]:
    if test_key not in datasets:
        print(f'[SKIP] {test_key} không có trong datasets')
        continue

    X_test = datasets[test_key]['X']
    y_test = datasets[test_key]['y']
    print(f'=== Test set: {test_key} — {X_test.shape} ===')
    print(f'{"Train size":>12}  {"F1-macro":>10}  {"Accuracy":>10}  {"Time":>8}')
    print('-' * 50)

    for n in TRAIN_SIZES:
        key = f'train_{n}'
        if key not in datasets:
            print(f'  [SKIP] {key} không tồn tại')
            continue

        t0  = time.time()
        svm = SVC(kernel='rbf', C=10.0, gamma='scale', random_state=RANDOM_STATE)
        svm.fit(datasets[key]['X'], datasets[key]['y'])
        y_pred  = svm.predict(X_test)
        elapsed = time.time() - t0

        f1_  = f1_score(y_test, y_pred, average='macro')
        acc_ = accuracy_score(y_test, y_pred)

        # Chỉ lưu kết quả test_100 vào svm_baselines để so sánh với QSVM
        if test_key == TEST_KEY_FIXED:
            svm_baselines[n] = {'f1': f1_, 'acc': acc_, 'time': elapsed, 'model': svm}

        print(f'{n:>12}  {f1_:>10.4f}  {acc_:>10.4f}  {elapsed:>7.2f}s')
    print()

svm_baseline_df = pd.DataFrame([
    {'train_size': n, 'f1': v['f1'], 'acc': v['acc'], 'time': v['time']}
    for n, v in svm_baselines.items()
])
print('SVM baselines sẵn sàng để so sánh với QSVM theo từng train size.')


## 6. QSVM Noiseless Baseline

In [ ]:
# Load hoặc Train QSVM noiseless cho mỗi size trong QSVM_TRAIN_SIZES
# Model được lưu tại: QSVM_MODEL_DIR/qsvm_noiseless_{n}.joblib
# Nếu file đã tồn tại → load, không train lại

qsvm_models  = {}   # {n: qsvc_object}
qsvm_results = {}   # {n: {'f1', 'acc', 'y_pred'}}

for n in QSVM_TRAIN_SIZES:
    key        = f'train_{n}'
    model_path = f'{QSVM_MODEL_DIR}/qsvm_noiseless_{n}.joblib'

    if key not in datasets:
        print(f'[SKIP] {key} không có trong datasets')
        continue

    X_tr = datasets[key]['X']
    y_tr = datasets[key]['y']

    if os.path.exists(model_path):
        print(f'[LOAD] qsvm_noiseless_{n}.joblib — bỏ qua train')
        qsvc = joblib.load(model_path)
    else:
        print(f'[TRAIN] QSVM noiseless n={n} ...')
        t0        = time.time()
        kernel_nl = build_qsvm_kernel(noise_model=None, shots=1024)
        qsvc      = QSVC(quantum_kernel=kernel_nl, C=1.0, random_state=RANDOM_STATE)
        qsvc.fit(X_tr, y_tr)
        elapsed = time.time() - t0
        joblib.dump(qsvc, model_path)
        print(f'  Train xong: {elapsed:.1f}s → lưu {model_path}')

    y_pred = qsvc.predict(X_test_fixed)
    f1_    = f1_score(y_test_fixed, y_pred, average='macro')
    acc_   = accuracy_score(y_test_fixed, y_pred)

    qsvm_models[n]  = qsvc
    qsvm_results[n] = {'f1': f1_, 'acc': acc_, 'y_pred': y_pred}

print()
print(f'{"Train size":>12}  {"F1-macro":>10}  {"Accuracy":>10}  {"SVM ref F1":>12}  {"Delta":>8}')
print('-' * 60)
for n, res in qsvm_results.items():
    svm_f1 = svm_baselines[n]['f1'] if n in svm_baselines else float('nan')
    delta  = res['f1'] - svm_f1
    print(f'{n:>12}  {res["f1"]:>10.4f}  {res["acc"]:>10.4f}  {svm_f1:>12.4f}  {delta:>+8.4f}')

# ── Reference variables cho các phân tích sau ──
train_n          = max(qsvm_results.keys())
QSVM_TRAIN_KEY   = f'train_{train_n}'
X_q_train        = datasets[QSVM_TRAIN_KEY]['X']
y_q_train        = datasets[QSVM_TRAIN_KEY]['y']
X_q_test         = X_test_fixed
y_q_test         = y_test_fixed
qsvc_noiseless   = qsvm_models[train_n]
y_pred_noiseless = qsvm_results[train_n]['y_pred']
f1_noiseless     = qsvm_results[train_n]['f1']
acc_noiseless    = qsvm_results[train_n]['acc']
svm_ref          = svm_baselines[train_n]

print(f'\nReference cho phân tích tiếp theo: train_n={train_n}')
print(f'  QSVM F1 = {f1_noiseless:.4f} | SVM F1 = {svm_ref["f1"]:.4f}')
print(f'  Support vectors: {qsvc_noiseless.n_support_} | Total: {qsvc_noiseless.n_support_.sum()}')


## 7. Depolarizing Noise Scan — Graceful Degradation

In [ ]:
depol_results = []

print(f'Quét {len(NOISE_LEVELS)} mức nhiễu depolarizing (train_n={train_n})...')
print(f'{"p1q":>8}  {"p2q":>8}  {"F1":>8}  {"Acc":>8}  {"Status":>10}')
print('-' * 55)

for p1q in NOISE_LEVELS:
    if p1q == 0.0:
        depol_results.append({'p1q': 0.0, 'p2q': 0.0,
                               'f1': f1_noiseless, 'acc': acc_noiseless, 'time': 0.0})
        print(f'{0.0:>8.3f}  {0.0:>8.3f}  {f1_noiseless:>8.4f}  '
              f'{acc_noiseless:>8.4f}  [noiseless]')
        continue

    p2q        = min(5 * p1q, 1.0)
    p_tag      = str(p1q).replace('.', 'p')
    model_path = f'{QSVM_MODEL_DIR}/qsvm_depol_{p_tag}_n{train_n}.joblib'

    if os.path.exists(model_path):
        qsvc_d = joblib.load(model_path)
        status = 'loaded'
        t_used = 0.0
    else:
        nm       = build_noise_model('depolarizing', p1q=p1q, p2q=p2q)
        t0       = time.time()
        kernel_d = build_qsvm_kernel(noise_model=nm, shots=1024)
        qsvc_d   = QSVC(quantum_kernel=kernel_d, C=1.0, random_state=RANDOM_STATE)
        qsvc_d.fit(X_q_train, y_q_train)
        t_used = time.time() - t0
        joblib.dump(qsvc_d, model_path)
        status = f'trained {t_used:.1f}s'

    y_pred = qsvc_d.predict(X_q_test)
    f1_    = f1_score(y_q_test, y_pred, average='macro')
    acc_   = accuracy_score(y_q_test, y_pred)
    depol_results.append({'p1q': p1q, 'p2q': p2q, 'f1': f1_, 'acc': acc_, 'time': t_used})
    print(f'{p1q:>8.3f}  {p2q:>8.3f}  {f1_:>8.4f}  {acc_:>8.4f}  [{status}]')

depol_df = pd.DataFrame(depol_results)


## 8. Amplitude Damping & Readout Error

In [ ]:
# ── Amplitude Damping ──
AD_LEVELS  = [0.0, 0.01, 0.02, 0.05, 0.10]
ad_results = []

print(f'Quét Amplitude Damping (train_n={train_n})...')
print(f'{"gamma":>8}  {"F1":>8}  {"Acc":>8}  {"Status":>10}')
print('-' * 40)

for gamma in AD_LEVELS:
    if gamma == 0.0:
        ad_results.append({'gamma': 0.0, 'f1': f1_noiseless, 'acc': acc_noiseless})
        print(f'{0.0:>8.3f}  {f1_noiseless:>8.4f}  {acc_noiseless:>8.4f}  [noiseless]')
        continue

    g_tag      = str(gamma).replace('.', 'p')
    model_path = f'{QSVM_MODEL_DIR}/qsvm_ad_{g_tag}_n{train_n}.joblib'

    if os.path.exists(model_path):
        q_ = joblib.load(model_path); status = 'loaded'
    else:
        nm = build_noise_model('amplitude_damping', gamma_ad=gamma)
        k_ = build_qsvm_kernel(noise_model=nm, shots=1024)
        q_ = QSVC(quantum_kernel=k_, C=1.0, random_state=RANDOM_STATE)
        q_.fit(X_q_train, y_q_train)
        joblib.dump(q_, model_path); status = 'trained'

    yp   = q_.predict(X_q_test)
    f1_  = f1_score(y_q_test, yp, average='macro')
    acc_ = accuracy_score(y_q_test, yp)
    ad_results.append({'gamma': gamma, 'f1': f1_, 'acc': acc_})
    print(f'{gamma:>8.3f}  {f1_:>8.4f}  {acc_:>8.4f}  [{status}]')

ad_df = pd.DataFrame(ad_results)

# ── Readout Error ──
RO_LEVELS  = [0.0, 0.01, 0.02, 0.05, 0.10]
ro_results = []

print(f'\nQuét Readout Error (train_n={train_n})...')
print(f'{"p_ro":>8}  {"F1":>8}  {"Acc":>8}  {"Status":>10}')
print('-' * 40)

for p_ro in RO_LEVELS:
    if p_ro == 0.0:
        ro_results.append({'p_ro': 0.0, 'f1': f1_noiseless, 'acc': acc_noiseless})
        print(f'{0.0:>8.3f}  {f1_noiseless:>8.4f}  {acc_noiseless:>8.4f}  [noiseless]')
        continue

    r_tag      = str(p_ro).replace('.', 'p')
    model_path = f'{QSVM_MODEL_DIR}/qsvm_ro_{r_tag}_n{train_n}.joblib'

    if os.path.exists(model_path):
        q_ = joblib.load(model_path); status = 'loaded'
    else:
        nm = build_noise_model('readout', readout_err=p_ro)
        k_ = build_qsvm_kernel(noise_model=nm, shots=1024)
        q_ = QSVC(quantum_kernel=k_, C=1.0, random_state=RANDOM_STATE)
        q_.fit(X_q_train, y_q_train)
        joblib.dump(q_, model_path); status = 'trained'

    yp   = q_.predict(X_q_test)
    f1_  = f1_score(y_q_test, yp, average='macro')
    acc_ = accuracy_score(y_q_test, yp)
    ro_results.append({'p_ro': p_ro, 'f1': f1_, 'acc': acc_})
    print(f'{p_ro:>8.3f}  {f1_:>8.4f}  {acc_:>8.4f}  [{status}]')

ro_df = pd.DataFrame(ro_results)


## 9. Zero-Noise Extrapolation (ZNE)

In [ ]:
zne_f1_observed = []

print(f'ZNE với p1q_base={ZNE_P1Q}, scales={ZNE_SCALES} (train_n={train_n})')
print(f'{"scale":>8}  {"p1q_eff":>10}  {"F1":>8}  {"Status":>10}')
print('-' * 42)

for scale in ZNE_SCALES:
    p1q_eff    = ZNE_P1Q * scale
    p2q_eff    = min(5 * p1q_eff, 1.0)
    model_path = f'{QSVM_MODEL_DIR}/qsvm_zne_s{scale}_n{train_n}.joblib'

    if os.path.exists(model_path):
        q_ = joblib.load(model_path); status = 'loaded'
    else:
        nm = build_noise_model('depolarizing', p1q=p1q_eff, p2q=p2q_eff)
        k_ = build_qsvm_kernel(noise_model=nm, shots=1024)
        q_ = QSVC(quantum_kernel=k_, C=1.0, random_state=RANDOM_STATE)
        q_.fit(X_q_train, y_q_train)
        joblib.dump(q_, model_path); status = 'trained'

    yp  = q_.predict(X_q_test)
    f1_ = f1_score(y_q_test, yp, average='macro')
    zne_f1_observed.append(f1_)
    print(f'{scale:>8}  {p1q_eff:>10.4f}  {f1_:>8.4f}  [{status}]')

scales_arr    = np.array(ZNE_SCALES, dtype=float)
f1_arr        = np.array(zne_f1_observed)
coeffs        = np.polyfit(scales_arr, f1_arr, deg=1)
f1_zne_extrap = np.polyval(coeffs, 0.0)

print(f'\nLinear ZNE extrapolated F1 (scale→0): {f1_zne_extrap:.4f}')
print(f'Noiseless F1 (reference)             : {f1_noiseless:.4f}')
print(f'Delta (ZNE - noiseless)              : {f1_zne_extrap - f1_noiseless:+.4f}')

zne_results = {
    'scales': ZNE_SCALES, 'f1_observed': zne_f1_observed,
    'f1_extrap': f1_zne_extrap, 'p1q_base': ZNE_P1Q
}


## 10. Per-Category Analysis

Chạy sau khi đã có đủ noisy models từ các bước trên — chỉ load từ cache, không train thêm.

In [ ]:
# ── Hàm phân tích per-category ──
def per_category_report(y_true, y_pred, meta_test, label=''):
    """Tính F1 theo attack_category."""
    cats    = meta_test['attack_category'].values
    results = []
    for cat in sorted(set(cats)):
        mask = cats == cat
        yt, yp = y_true[mask], y_pred[mask]
        n = mask.sum()
        if len(set(yt)) < 2:
            f1_ = (yt == yp).mean()
        else:
            f1_ = f1_score(yt, yp, average='macro', zero_division=0)
        results.append({'category': cat, 'n': n, 'f1': f1_, 'label': label})
    return pd.DataFrame(results)


test_meta_fixed = datasets[TEST_KEY_FIXED]['meta'].reset_index(drop=True)
cat_list        = []

# SVM baselines per-category
for n, res_svm in svm_baselines.items():
    y_pred_svm_n = res_svm['model'].predict(X_test_fixed)
    cat_list.append(per_category_report(y_test_fixed, y_pred_svm_n,
                                        test_meta_fixed, label=f'SVM n={n}'))

# QSVM noiseless per-category
for n, res in qsvm_results.items():
    cat_list.append(per_category_report(y_test_fixed, res['y_pred'],
                                        test_meta_fixed, label=f'QSVM n={n}'))

# QSVM noisy per-category — chỉ load từ cache (đã train ở bước 7)
print(f'Load QSVM noisy models cho per-category (NOISE_REPR={NOISE_REPR})...')
noisy_cat_preds = {}
for p1q in NOISE_REPR:
    p_tag      = str(p1q).replace('.', 'p')
    model_path = f'{QSVM_MODEL_DIR}/qsvm_depol_{p_tag}_n{train_n}.joblib'
    if os.path.exists(model_path):
        q_  = joblib.load(model_path)
        yp  = q_.predict(X_q_test)
        noisy_cat_preds[p1q] = yp
        cat_list.append(per_category_report(y_q_test, yp, test_meta_fixed,
                                            label=f'QSVM p={p1q:.2f}'))
        print(f'  [LOAD] p1q={p1q}')
    else:
        print(f'  [MISSING] p1q={p1q} — chạy lại bước 7 trước')

cat_all_df = pd.concat(cat_list, ignore_index=True)
pivot_f1   = cat_all_df.pivot(index='category', columns='label', values='f1').round(4)
pivot_n    = cat_all_df[cat_all_df['label'] == f'QSVM n={train_n}'][['category', 'n']].set_index('category')

print()
print('=== F1-macro per attack_category ===')
print(pivot_f1.to_string())
print()
print('Phân bố mẫu test theo category:')
print(pivot_n.to_string())


## 11. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Contribution 3: Đánh Giá Độ Bền Nhiễu NISQ — NSL-KDD\n'
             f'QSVM 4-qubit ZZFeatureMap (reps={REPS}) | '
             f'Train={X_q_train.shape[0]} ({QSVM_TRAIN_KEY}) | Test={X_q_test.shape[0]} ({TEST_KEY_FIXED})',
             fontsize=13, fontweight='bold')

svm_ref_f1 = svm_baselines[train_n]['f1']

# ── Plot 1: Depolarizing — graceful degradation + SVM reference ──
ax = axes[0, 0]
ax.plot(depol_df['p1q'] * 100, depol_df['f1'], 'o-',
        color='#1565C0', linewidth=2.5, markersize=8, label='QSVM (ZZFeatureMap)')
ax.axhline(svm_ref_f1, color='#E53935', linestyle='--', linewidth=2,
           label=f'SVM RBF (train={train_n}, F1={svm_ref_f1:.4f})')
ax.axhline(f1_noiseless, color='#2E7D32', linestyle=':', linewidth=1.8,
           label=f'QSVM noiseless ({f1_noiseless:.4f})')
ax.scatter([ZNE_P1Q * 100], [zne_results['f1_extrap']],
           marker='*', s=250, color='orange', zorder=6,
           label=f'ZNE extrap ({zne_results["f1_extrap"]:.4f})')
ax.set_xlabel('Depolarizing error rate p1q (%)', fontsize=11)
ax.set_ylabel('F1-macro', fontsize=11)
ax.set_title('Graceful Degradation — Depolarizing Noise\n(so sánh cùng train size)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.05)

# ── Plot 2: SVM vs QSVM theo từng train size ──
ax2 = axes[0, 1]
sizes       = svm_baseline_df['train_size'].values
f1_svm_arr  = svm_baseline_df['f1'].values
x = np.arange(len(sizes))
w = 0.35
ax2.bar(x - w/2, f1_svm_arr, w, color='#E53935', alpha=0.85,
        edgecolor='white', label='Classical SVM RBF')
ax2.bar(x + w/2, [f1_noiseless if s == train_n else np.nan for s in sizes],
        w, color='#1565C0', alpha=0.85, edgecolor='white', label=f'QSVM noiseless (train={train_n})')
for i, (s, f) in enumerate(zip(sizes, f1_svm_arr)):
    ax2.text(i - w/2, f + 0.005, f'{f:.3f}', ha='center', fontsize=8)
if train_n in sizes.tolist():
    idx = sizes.tolist().index(train_n)
    ax2.text(idx + w/2, f1_noiseless + 0.005, f'{f1_noiseless:.3f}', ha='center', fontsize=8, color='#1565C0')
ax2.set_xticks(x); ax2.set_xticklabels([str(s) for s in sizes])
ax2.set_xlabel('Train size', fontsize=11); ax2.set_ylabel('F1-macro', fontsize=11)
ax2.set_title('SVM vs QSVM — Fair Comparison (cùng train size, cùng test)', fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3, axis='y'); ax2.set_ylim(0, 1.1)

# ── Plot 3: So sánh 3 loại nhiễu ──
ax3 = axes[1, 0]
x_depol = depol_df['p1q'].values
x_ad    = ad_df['gamma'].values
x_ro    = ro_df['p_ro'].values
ax3.plot(x_depol / x_depol.max(), depol_df['f1'], 'o-',
         color='#1565C0', linewidth=2, label='Depolarizing (p1q)')
ax3.plot(x_ad / x_ad.max(), ad_df['f1'], 's--',
         color='#E65100', linewidth=2, label='Amplitude Damping (γ)')
ax3.plot(x_ro / x_ro.max(), ro_df['f1'], '^:',
         color='#6A1B9A', linewidth=2, label='Readout Error (p_ro)')
ax3.axhline(f1_noiseless, color='gray', linestyle='-.', linewidth=1.5, label='Noiseless baseline')
ax3.set_xlabel('Mức nhiễu (chuẩn hóa 0→1)', fontsize=11)
ax3.set_ylabel('F1-macro', fontsize=11)
ax3.set_title('So Sánh 3 Loại Nhiễu', fontweight='bold')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3); ax3.set_ylim(0, 1.05)

# ── Plot 4: ZNE extrapolation ──
ax4 = axes[1, 1]
scale_fine = np.linspace(0, max(ZNE_SCALES) + 0.5, 100)
f1_fit     = np.polyval(coeffs, scale_fine)
ax4.plot(ZNE_SCALES, zne_results['f1_observed'], 'o',
         color='#1565C0', markersize=10, zorder=5, label='Quan sát')
ax4.plot(scale_fine, f1_fit, '--', color='#1565C0', linewidth=2, label='Linear fit')
ax4.scatter([0], [zne_results['f1_extrap']], marker='*', s=350,
            color='orange', zorder=6, label=f'ZNE: {zne_results["f1_extrap"]:.4f}')
ax4.scatter([0], [f1_noiseless], marker='D', s=150,
            color='#2E7D32', zorder=6, label=f'Noiseless: {f1_noiseless:.4f}')
ax4.axvline(x=0, color='gray', linestyle=':', linewidth=1.5)
ax4.set_xlabel('Noise scale factor', fontsize=11); ax4.set_ylabel('F1-macro', fontsize=11)
ax4.set_title(f'Zero-Noise Extrapolation (ZNE)\np1q_base={ZNE_P1Q}', fontweight='bold')
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.3)
ax4.set_xlim(-0.3, max(ZNE_SCALES) + 0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/c3_noise_robustness.png', dpi=150, bbox_inches='tight')
plt.show()
print('Đã lưu: c3_noise_robustness.png')


## 11b. Visualization Per-Category

In [ ]:
# ── Heatmap F1 theo category × noise level ──
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle('Phân Tích Per-Category — C3 Noise Robustness\n'
             f'Train={train_n} | Test={TEST_KEY_FIXED} | 4-qubit ZZFeatureMap',
             fontsize=13, fontweight='bold')

ax = axes[0]
col_order  = ([f'SVM n={train_n}', f'QSVM n={train_n}'] +
              [f'QSVM p={p:.2f}' for p in NOISE_REPR])
pivot_plot = pivot_f1[[c for c in col_order if c in pivot_f1.columns]]

sns.heatmap(pivot_plot, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'F1-macro'})
ax.set_title('F1-macro per Attack Category\nvs Noise Level', fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('Attack Category')
ax.tick_params(axis='x', rotation=30)

ax2       = axes[1]
cats      = pivot_plot.index.tolist()
n_configs = len(pivot_plot.columns)
x         = np.arange(len(cats))
width     = 0.8 / n_configs

colors_bar = ['#E53935', '#2E7D32', '#1976D2', '#F57C00', '#6A1B9A']
for j, col in enumerate(pivot_plot.columns):
    offset = (j - n_configs / 2 + 0.5) * width
    ax2.bar(x + offset, pivot_plot[col].values, width, label=col,
            color=colors_bar[j % len(colors_bar)], alpha=0.85, edgecolor='white')

ax2.set_xticks(x)
ax2.set_xticklabels(cats, rotation=15, ha='right')
ax2.set_ylabel('F1-macro', fontsize=11)
ax2.set_title('F1 per Category — SVM vs QSVM vs Noisy\n(U2R & R2L: rare attacks)',
              fontweight='bold')
ax2.legend(fontsize=8, loc='lower right')
ax2.set_ylim(0, 1.15)
ax2.grid(True, alpha=0.3, axis='y')

for label in ax2.get_xticklabels():
    if label.get_text() in ('U2R', 'R2L'):
        label.set_color('#C62828')
        label.set_fontweight('bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/c3_per_category_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Đã lưu: c3_per_category_analysis.png')


## 12. Lưu Kết Quả

In [ ]:
depol_df.to_csv(f'{OUTPUT_DIR}/c3_depolarizing_results.csv', index=False)
ad_df.to_csv(f'{OUTPUT_DIR}/c3_amplitude_damping_results.csv', index=False)
ro_df.to_csv(f'{OUTPUT_DIR}/c3_readout_results.csv', index=False)

zne_summary = pd.DataFrame({
    'scale': ZNE_SCALES,
    'f1_observed': zne_results['f1_observed'],
    'f1_extrap_linear': [zne_results['f1_extrap']] * len(ZNE_SCALES),
    'f1_noiseless': [f1_noiseless] * len(ZNE_SCALES),
    'p1q_base': ZNE_P1Q
})
zne_summary.to_csv(f'{OUTPUT_DIR}/c3_zne_results.csv', index=False)

cat_all_df.to_csv(f'{OUTPUT_DIR}/c3_per_category_results.csv', index=False)
pivot_f1.to_csv(f'{OUTPUT_DIR}/c3_per_category_pivot.csv')

print('=== ĐÃ LƯU ===')
for fname, info in [
    ('c3_depolarizing_results.csv',      f'{len(depol_df)} mức nhiễu'),
    ('c3_amplitude_damping_results.csv', f'{len(ad_df)} mức gamma'),
    ('c3_readout_results.csv',           f'{len(ro_df)} mức readout error'),
    ('c3_zne_results.csv',               f'{len(ZNE_SCALES)} scale factors'),
    ('c3_per_category_results.csv',      f'{len(cat_all_df)} rows'),
    ('c3_per_category_pivot.csv',        'pivot table'),
    ('c3_noise_robustness.png',          'Visualization 4 plots'),
    ('c3_per_category_analysis.png',     'Per-category heatmap & bar'),
]:
    print(f'  {fname:<45} {info}')


## 13. Summary — Contribution 3

In [ ]:
valid_mask       = depol_df['p1q'] > 0
slope_depol      = np.polyfit(depol_df.loc[valid_mask, 'p1q'],
                               depol_df.loc[valid_mask, 'f1'], 1)[0]
f1_at_max_noise  = depol_df['f1'].iloc[-1]
f1_drop_pct      = (f1_noiseless - f1_at_max_noise) / f1_noiseless * 100

print('=' * 65)
print('  CONTRIBUTION 3 — NOISE ROBUSTNESS EVALUATION (NSL-KDD)')
print('=' * 65)
print(f'  Circuit          : ZZFeatureMap, {N_QUBITS} qubits, reps={REPS}')
print(f'  QSVM train       : {X_q_train.shape[0]} ({QSVM_TRAIN_KEY}) | Test: {X_q_test.shape[0]} ({TEST_KEY_FIXED})')
print()
print(f'  [Fair Comparison — cùng train size = {train_n}, cùng test]')
for n, v in svm_baselines.items():
    marker = ' <- QSVM ref' if n == train_n else ''
    print(f'    SVM RBF train={n:<5} : F1 = {v["f1"]:.4f}{marker}')
print(f'    QSVM Noiseless       : F1 = {f1_noiseless:.4f}')
print(f'    Delta QSVM - SVM     : {f1_noiseless - svm_baselines[train_n]["f1"]:+.4f}')
print()
print(f'  [Depolarizing Noise — p1q=0→{NOISE_LEVELS[-1]}, p2q=5×p1q]')
print(f'    F1 at p={NOISE_LEVELS[-1]:.2f}    : {f1_at_max_noise:.4f}')
print(f'    F1 drop          : {f1_drop_pct:.1f}%  → graceful degradation')
print(f'    Slope            : {slope_depol:.4f} F1/unit_noise')
print()
print(f'  [Amplitude Damping — gamma=0→0.10]')
for r in ad_df.itertuples():
    print(f'    gamma={r.gamma:.2f}  F1={r.f1:.4f}')
print()
print(f'  [Readout Error — p_ro=0→0.10]')
for r in ro_df.itertuples():
    print(f'    p_ro ={r.p_ro:.2f}  F1={r.f1:.4f}')
print()
print(f'  [Zero-Noise Extrapolation — p1q_base={ZNE_P1Q}]')
print(f'    Scales    : {ZNE_SCALES}')
print(f'    F1 obs    : {[f"{v:.4f}" for v in zne_results["f1_observed"]]}')
print(f'    ZNE extrap: {zne_results["f1_extrap"]:.4f}  (Richardson linear)')
print(f'    Delta vs noiseless : {zne_results["f1_extrap"] - f1_noiseless:+.4f}')
print('=' * 65)
